# Clase 4 — Chain-of-thought y prompts complejos

Hay tareas que requieren razonamiento en varios pasos: resolver un problema, tomar una decisión con múltiples criterios o analizar una situación con información incompleta. En esos casos, pedirle al modelo que piense en voz alta — **chain-of-thought** — mejora significativamente la precisión.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Qué es chain-of-thought y por qué funciona |
| 3 | CoT implícito vs. CoT explícito |
| 4 | CoT estructurado con pasos definidos |
| 5 | Restricciones y rúbricas dentro del prompt |
| 6 | Actividad: optimizar un prompt complejo |

---
## 1. Configuración del entorno

**Si es tu primera vez en este curso:**
1. Obtené tu API key en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. Guardala en `.env`:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. Si no querés crear el archivo, la celda te la pide de forma interactiva.

In [4]:
import os
import getpass

BACKEND = "ollama"         # "gemini", "ollama", "local"
GEMINI_MODEL = "gemini-2.5-flash-lite"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

Backend: ollama


In [5]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

elif BACKEND == "ollama":
    import ollama
    OLLAMA_MODEL = "gemma2:9b"  # Modelo que tenés descargado
    print("🚀 Conectando a Ollama...")
    try:
        ollama.list()
        print(f"✅ Ollama disponible. Usando modelo: {OLLAMA_MODEL}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Solución: Abre otra terminal y ejecuta: ollama serve")
        raise

elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        filename="qwen2.5-0.5b-instruct-q4_k_m.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=300):
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    elif BACKEND == "ollama":
        r = ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            system=system_prompt,
            stream=False,
            options={
                "temperature": temperature,
                "num_predict": max_tokens,
            }
        )
        return r['response'].strip()
    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

🚀 Conectando a Ollama...
✅ Ollama disponible. Usando modelo: gemma2:9b
Entorno listo.


---
## 2. Qué es chain-of-thought y por qué funciona

Cuando un modelo responde directamente una pregunta compleja, puede saltarse pasos lógicos y llegar a una conclusión incorrecta. **Chain-of-thought (CoT)** es la técnica de pedirle que muestre su razonamiento paso a paso antes de dar la respuesta final.

Funciona porque:
- El modelo genera cada token en función de los anteriores. Si los pasos intermedios están escritos, el siguiente token es más probable que sea correcto.
- El razonamiento visible también te permite detectar dónde se equivocó.

| Tipo de tarea | ¿Vale la pena usar CoT? |
|---|---|
| Saludo, formato simple, resumen breve | No — el overhead no aporta |
| Razonamiento lógico o matemático | Sí — mejora mucho la precisión |
| Decisiones con múltiples criterios | Sí — fuerza a ponderar antes de concluir |
| Análisis de situaciones ambiguas | Sí — reduce conclusiones apresuradas |

---
## 3. CoT implícito vs. CoT explícito

Hay dos formas de activar el razonamiento paso a paso:

In [6]:
# ─── Tarea de referencia ─────────────────────────────────────────────────────
# Una empresa tiene tres candidatos para un puesto. Necesita elegir uno
# considerando: experiencia, disponibilidad y presupuesto.

caso = """Una empresa necesita contratar un analista de datos.
Tiene tres candidatos:
- Ana: 5 años de experiencia, disponible en 2 semanas, pide $2.500/mes.
- Bruno: 2 años de experiencia, disponible de inmediato, pide $1.800/mes.
- Luis: 5 años de experiencia, disponible en 1 semana, pide $2.000/mes.
- Carmen: 8 años de experiencia, disponible en 2 meses, pide $3.200/mes.
El proyecto empieza en 3 semanas y el presupuesto máximo es $2.800/mes."""

pregunta = "¿A quién deberían contratar y por qué?"

# ─── Sin CoT: respuesta directa ──────────────────────────────────────────────
print("=== SIN CoT ===")
print(llamar_llm(f"{caso}\n\n{pregunta}", max_tokens=1000))
print()

=== SIN CoT ===
Dada la información, la mejor opción sería **Luis**.  

Aquí está el análisis:

* **Experiencia:** Luis tiene 5 años de experiencia, lo que es comparable a Ana. Bruno tiene menos experiencia. Carmen tiene más, pero su alto salario podría exceder el presupuesto.
* **Disponibilidad:** Luis está disponible en una semana, coincidiendo con el inicio del proyecto en 3 semanas. Ana está disponible en 2 semanas y Bruno inmediatamente. Carmen no estaría disponible hasta dentro de 2 meses.
* **Salario:** Luis pide $2.000/mes, lo que se ajusta al presupuesto máximo de $2.800/mes.

Aunque Ana también tiene 5 años de experiencia, Luis es la mejor opción por su disponibilidad y salario competitivo.



In [7]:
# ─── CoT implícito: la frase mágica ──────────────────────────────────────────
# Solo agregar "Pensá paso a paso" activa el razonamiento en muchos casos.

print("=== CoT IMPLÍCITO ===")
prompt_cot_implicito = f"{caso}\n\n{pregunta}\n\nPensá paso a paso antes de responder."
print(llamar_llm(prompt_cot_implicito, max_tokens=1000))
print()

=== CoT IMPLÍCITO ===
Aquí está el análisis paso a paso:

1. **Prioridades:** La empresa necesita un analista de datos disponible en 3 semanas, dentro del presupuesto de $2.800/mes. 

2. **Candidatos que cumplen con la disponibilidad:** Ana (2 semanas) y Luis (1 semana) son candidatos disponibles a tiempo. Bruno (inmediatamente) también cumple.

3. **Comparación de salarios:**
    * Ana: $2.500/mes - Excede el presupuesto.
    * Bruno: $1.800/mes - Dentro del presupuesto.
    * Luis: $2.000/mes - Dentro del presupuesto.
    * Carmen: $3.200/mes - Excede el presupuesto y no está disponible a tiempo.

4. **Experiencia:** Aunque la experiencia es importante, la disponibilidad e inversión son cruciales en este caso.

**Conclusión:** La mejor opción para la empresa es **Luis**. Cumple con ambas necesidades clave: disponibilidad dentro del plazo (1 semana) y salario dentro del presupuesto ($2.000/mes).



In [12]:
# ─── CoT explícito: pasos definidos por nosotros ─────────────────────────────
# Le indicamos exactamente qué analizar en cada paso.

print("=== CoT EXPLÍCITO ===")
prompt_cot_explicito = f"""{caso}

Para elegir al candidato, seguí estos pasos en orden:
Paso 1: Elimina a los candidatos que piden mas que el presupuesto.
Paso 1.1: lista los que que piden menos que el presupuesto.
Paso 2: De la lista previamente filtrada, descartá a quienes no pueden empezar a tiempo.
Paso 2.1: lista los que pueden empezar a tiempo.
Paso 3: De los que quedan, elegí el de mayor experiencia.
Paso 4: Escribí la recomendación final que cumpla con los requerimientosen una oración."""

print(llamar_llm(prompt_cot_explicito, max_tokens=300))

=== CoT EXPLÍCITO ===
## Selección del analista de datos:

**Paso 1:**  Los candidatos que piden menos que el presupuesto de $2.800/mes son: Bruno ($1.800/mes), Luis ($2.000/mes).

**Paso 2:** De la lista anterior, los que pueden empezar a tiempo (en 3 semanas) son: Bruno y Luis.

**Paso 3:**  El candidato con mayor experiencia entre Bruno y Luis es Luis (5 años de experiencia).

**Paso 4:** Se recomienda contratar a **Luis**, ya que cumple con el presupuesto, la disponibilidad y tiene mayor experiencia.


> 💡 **Para discutir:** ¿Cuál de los tres enfoques fue más útil para este caso? ¿En qué tipo de decisiones de tu trabajo aplicarías CoT explícito?

---
## 4. CoT estructurado con pasos definidos

Cuando la tarea se repite muchas veces (por ejemplo, analizar contratos, evaluar propuestas, revisar reportes), conviene estandarizar los pasos del CoT en una plantilla reutilizable.

In [13]:
# ─── Plantilla CoT para análisis de riesgo ───────────────────────────────────
# Caso de uso: evaluar si un proveedor nuevo es confiable.

def analizar_proveedor_cot(descripcion_proveedor):
    prompt = f"""Sos un analista de compras evaluando proveedores nuevos.

Información del proveedor:
{descripcion_proveedor}

Analizá siguiendo estos pasos:
Paso 1 — Experiencia: ¿Cuántos años opera y en qué industrias?
Paso 2 — Capacidad: ¿Puede cumplir el volumen y los plazos requeridos?
Paso 3 — Riesgos: Identificá hasta 2 riesgos concretos.
Paso 4 — Recomendación: Aprobado / Aprobado con condiciones / Rechazado + una oración de justificación.

Mostrá cada paso claramente numerado."""

    return llamar_llm(prompt, max_tokens=350)


proveedor_ejemplo = """TechSupply S.A. lleva 3 años en el mercado de insumos electrónicos.
Tiene capacidad para entregar 500 unidades mensuales con un plazo de 15 días.
No tiene certificaciones de calidad pero sí referencias de dos clientes medianos.
Su precio es un 20% menor al mercado actual."""

print(analizar_proveedor_cot(proveedor_ejemplo))

## Análisis de Proveedor: TechSupply S.A.

**Paso 1 — Experiencia:** TechSupply S.A. opera en el mercado de insumos electrónicos desde hace **3 años**.  No se menciona su experiencia en otras industrias.

**Paso 2 — Capacidad:** Puede cumplir con la capacidad requerida (500 unidades mensuales) y el plazo de entrega (15 días).

**Paso 3 — Riesgos:**
* **Falta de certificaciones de calidad:** La ausencia de certificaciones puede indicar un menor control de calidad en sus procesos.
* **Referencias limitadas:**  Depender únicamente de referencias de dos clientes medianos podría no ser suficiente para evaluar su fiabilidad y capacidad de servicio a gran escala.

**Paso 4 — Recomendación:** **Aprobado con condiciones.** Si bien TechSupply S.A. cumple con los requisitos básicos, se recomienda solicitar pruebas de calidad y ampliar las referencias a clientes de mayor tamaño antes de formalizar una compra significativa.


---
## 5. Restricciones y rúbricas dentro del prompt

Para casos donde la salida necesita cumplir criterios específicos, podemos incluir una **rúbrica** dentro del prompt: una lista explícita de condiciones que la respuesta debe satisfacer. El modelo la usa como checklist interno.

In [14]:
# ─── Prompt con rúbrica explícita ─────────────────────────────────────────────
# Tarea: escribir un mensaje de feedback para un empleado.

situacion = """Lucas entregó el informe dos días tarde, pero la calidad del trabajo
fue muy buena. Es la primera vez que se retrasa en 18 meses de trabajo."""

prompt_con_rubrica = f"""Sos un líder de equipo. Escribí un mensaje de feedback para Lucas sobre esta situación:
{situacion}

El mensaje debe cumplir TODOS estos criterios:
✓ Reconocer primero lo positivo antes de mencionar el problema.
✓ Mencionar el retraso como un hecho, sin juicio sobre la persona.
✓ Proponer una acción concreta para evitar que se repita.
✓ Tono constructivo y directo, sin condescendencia.
✓ Máximo 5 oraciones en total.

Antes del mensaje, verificá en una línea que cumplís cada criterio."""

print(llamar_llm(prompt_con_rubrica, max_tokens=350))

✔ Reconocer lo positivo / ✔ Mencionar el retraso como hecho / ✔ Acción concreta / ✔ Tono constructivo / ✔ 5 oraciones


Lucas, me gustó mucho la calidad del informe, es un trabajo excelente. No obstante, se entregó dos días después del plazo acordado.  Para evitar inconvenientes en el futuro, ¿puedes enviarme una notificación si te vas a retrasar con alguna entrega? De esta forma podemos ajustar las expectativas y seguir trabajando de manera eficiente.


In [15]:
# ─── Comparación: sin rúbrica vs. con rúbrica ─────────────────────────────────

print("=== SIN RÚBRICA ===")
print(llamar_llm(
    f"Sos un líder de equipo. Escribí un mensaje de feedback para Lucas: {situacion}",
    max_tokens=200
))
print()
print("=== CON RÚBRICA ===")
print(llamar_llm(prompt_con_rubrica, max_tokens=350))

=== SIN RÚBRICA ===
Lucas,

El informe es excelente, ¡el trabajo está realmente bien hecho!  

Sé que llegó un poco tarde, lo cual aprecio que hayas comunicado. Aunque esto sea la primera vez en 18 meses, es importante entregar los proyectos a tiempo. Hagamos una breve pausa para hablar sobre cómo podemos evitar esto en el futuro.

Gracias por tu dedicación al trabajo.

=== CON RÚBRICA ===
✓ Reconocer lo positivo  ✓ Mencionar el retraso como hecho ✓ Acción concreta ✓ Tono constructivo ✓ Máximo 5 oraciones


Lucas, el informe está excelente, la investigación y análisis son muy detallados. No obstante, se entregó dos días después del plazo acordado. Para evitar esto en el futuro, ¿te parece que nos ponemos de acuerdo para una notificación anticipada si surgen imprevistos?


---
## 6. Actividad: optimizar un prompt complejo

Tomá una tarea de razonamiento o decisión de tu trabajo. Escribí primero un prompt simple y luego una versión con CoT explícito + rúbrica. Compará ambas respuestas.

In [25]:
# TODO: Describí una situación que requiera razonamiento o decisión
mi_situacion = """Armando Esteban Quito es un agente de ventas que ha vendido en total 10 unidades, cerro 20 ventas, 30 leads"""

# ─── Versión simple ───────────────────────────────────────────────────────────
# TODO: Escribí un prompt directo sin CoT
mi_prompt_simple = """Realiza un informe de rendimiento del agente de ventas Armando Esteban Quito para gerencia del periodo abril - mayo."""

print("=== VERSIÓN SIMPLE ===")
print(llamar_llm(mi_prompt_simple, max_tokens=200))
print()

=== VERSIÓN SIMPLE ===
## Informe de Rendimiento: Armando Esteban Quito (Abril - Mayo)

**Periodo:** Abril - Mayo [Año]

**Vendedor:** Armando Esteban Quito

**Resumen:** 

Armando ha mostrado un desempeño **[positivo/negativo]** durante los meses de abril y mayo. 

**Puntos a destacar:**

* **Ventas totales:** [Cantidad total de ventas en dólares o unidades]. Este número se compara con [porcentaje de aumento/disminución] respecto al periodo anterior (marzo).
* **Principales productos vendidos:** [Lista de los 3 productos más vendidos por Armando durante el periodo].
* **Clientes nuevos obtenidos:** [Número de clientes nuevos que ha logrado atraer Armando].
* **Cumplimiento del objetivo de ventas:** [Indique si Armando cumplió, superó o no alcanzó su objetivo de ventas para el periodo].

**Análisis:**

* **Fortalezas:** [Mencione las fortalezas de Armando, como habilidades de comunicación, conocimiento del



In [26]:
# TODO: Reescribí el mismo prompt con pasos CoT explícitos y una rúbrica
mi_prompt_cot = """Realiza un informe de rendimiento de agente de ventas para gerencia. con análisis de tendencias, identificación de oportunidades y recomendaciones de mejora.

Seguí estos pasos:
Paso 1: Analizá las ventas del del equipo del periodo abril - mayo.
Paso 2: Identificá si el agente está por debajo o por encima del promedio del equipo.
Paso 3: Proponé recomendaciones concretas para mejorar el rendimiento del agente.

La respuesta debe cumplir:
✓  Comparar con el equipo para contextualizar el rendimiento.
✓  Identificar oportunidades de mejora basadas en las tendencias.
✓  Proponer recomendaciones concretas para mejorar el rendimiento del agente de ventas.
✓  Recomendaciones accionables y específicas."""

print("=== VERSIÓN CoT + RÚBRICA ===")
print(llamar_llm(mi_prompt_cot, max_tokens=350))

=== VERSIÓN CoT + RÚBRICA ===
## Informe de Rendimiento - Agente de Ventas (Abril - Mayo)

**Introducción:** Este informe analiza el desempeño del agente de ventas durante los meses de abril y mayo, comparándolo con el promedio del equipo para identificar áreas de mejora y oportunidades de crecimiento.

**Paso 1: Análisis de las ventas (Abril - Mayo)**

* **Ventas totales del agente:**  [Inserte aquí el total de ventas del agente en Abril y Mayo]
* **Promedio de ventas del equipo:** [Inserte aquí el promedio de ventas por miembro del equipo en Abril y Mayo]

**Paso 2: Rendimiento en relación al equipo:**

* El agente se encuentra [**por encima/por debajo**] del promedio del equipo en las ventas durante abril y mayo.  
    * **Por encima del promedio:**  El agente ha superado el rendimiento promedio, mostrando una capacidad sólida de cierre y generación de oportunidades.
    * **Por debajo del promedio:** El agente necesita reforzar sus habilidades para alcanzar el nivel del equipo.

**

---
## Entregable

Guardá el notebook con las celdas ejecutadas.
El entregable es la actividad de la sección 6: prompt original, versión CoT + rúbrica y comparación de resultados con un comentario breve sobre qué mejoró.

**Para la próxima clase:** vamos a ver cómo organizar y documentar prompts para que sean reutilizables en equipo.